# series/generative-adversarial-networks/01 GAN Architecture

Part 1 of the GAN series. Fresh Python 3.11+ kernel; CPU, no API keys or dataset required. Complete source architectures with repaired indentation, explicit inputs and current LeakyReLU parameter. Preserve the source final ReLU (not a tanh substitution). Original unmodified blocks remain in the appendix.

In [ ]:
import os
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "tensorflow==2.20.0", "keras==3.15.1", "numpy==2.4.6", "certifi==2026.10.23"], check=True)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Sequential, Input
from tensorflow.keras.layers import Conv2D, LeakyReLU, Dropout, ZeroPadding2D, BatchNormalization, Flatten, Dense, Reshape, UpSampling2D
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
keras.utils.set_random_seed(42)
np.set_printoptions(precision=6, suppress=True)
print("Python:", sys.version.split()[0], "| TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

## Example 0

In [ ]:
def discriminator_model():
    discriminator = Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, kernel_size=3, strides=2, padding="same"),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Conv2D(64, kernel_size=3, strides=2, padding="same"),
        ZeroPadding2D(padding=((0, 1), (0, 1))),
        BatchNormalization(momentum=0.8),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Conv2D(128, kernel_size=3, strides=2, padding="same"),
        BatchNormalization(momentum=0.8),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Conv2D(256, kernel_size=3, strides=1, padding="same"),
        BatchNormalization(momentum=0.8),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Flatten(), Dense(1, activation="sigmoid"),
    ], name="discriminator")
    return discriminator

discriminator = discriminator_model()
print("Discriminator input/output:", discriminator.input_shape, discriminator.output_shape)
print("Parameters:", discriminator.count_params())

## Example 1

In [ ]:
def generator_model():
    generator = Sequential([
        Input(shape=(100,)),
        Dense(128 * 7 * 7, activation="relu"),
        Reshape((7, 7, 128)), UpSampling2D(size=(2, 2)),
        Conv2D(128, kernel_size=3, padding="same", activation="relu"),
        BatchNormalization(momentum=0.8),
        UpSampling2D(size=(2, 2)),
        Conv2D(64, kernel_size=3, padding="same", activation="relu"),
        BatchNormalization(momentum=0.8),
        Conv2D(1, kernel_size=3, padding="same", activation="relu"),
    ], name="generator")
    return generator

generator = generator_model()
noise = np.random.normal(0, 1, (4, 100)).astype("float32")
images = generator(noise, training=False).numpy()
probabilities = discriminator(images, training=False).numpy()
print("Generator input/output:", generator.input_shape, generator.output_shape)
print("Parameters:", generator.count_params())
print("Generated batch:", images.shape)
print("Discriminator probabilities:")
print(probabilities)

## Independent regression checks

In [ ]:
assert discriminator.input_shape == (None, 28, 28, 1)
assert discriminator.output_shape == (None, 1)
assert generator.input_shape == (None, 100)
assert generator.output_shape == (None, 28, 28, 1)
assert images.shape == (4, 28, 28, 1) and np.isfinite(images).all()
assert (images >= 0).all()  # source output activation is ReLU
assert probabilities.shape == (4, 1) and ((probabilities >= 0) & (probabilities <= 1)).all()
assert [x.filters for x in discriminator.layers if isinstance(x, Conv2D)] == [32,64,128,256]
assert [x.filters for x in generator.layers if isinstance(x, Conv2D)] == [128,64,1]
assert sum(isinstance(x, UpSampling2D) for x in generator.layers) == 2
np.testing.assert_array_equal(images, generator(noise, training=False).numpy())
print("Architecture, layer order, shapes, finite values, sigmoid bounds and deterministic inference passed.")

## Original source appendix

Full original blocks from raw source SHA-256 29c444eb3ef22a794abc5efa318c6774f197900d2ad6451e15017bb6323bc6a7. Preserved verbatim as nonexecuting historical material, including mistakes/legacy setup alternatives.

### Original block 1

```text
def discriminator_model():
  
# instantiate a sequential model and name it discriminator
  discriminator = Sequential()
  
# add a convolutional layer to the discriminator model
  discriminator.add(Conv2D(32, kernel_size=3, strides=2, input_shape=(28,28,1),
  padding="same"))
  
# add a leakyRelu activation function
   discriminator.add(LeakyReLU(alpha=0.2))
  
# add a dropout layer with a 25% dropout probability
   discriminator.add(Dropout(0.25))
  
# add a second convolutional layer with zero padding
   discriminator.add(Conv2D(64, kernel_size=3, strides=2, padding="same"))
   discriminator.add(ZeroPadding2D(padding=((0,1),(0,1))))
  
# add a BatchNormalization layer for faster learning and higher accuracy
   discriminator.add(BatchNormalization(momentum=0.8))
   discriminator.add(LeakyReLU(alpha=0.2))
   discriminator.add(Dropout(0.25))
  
# add a third convolutional layer with batch norm, leakyRelu, and a dropout
   discriminator.add(Conv2D(128, kernel_size=3, strides=2, padding="same"))
   discriminator.add(BatchNormalization(momentum=0.8))
   discriminator.add(LeakyReLU(alpha=0.2))
   discriminator.add(Dropout(0.25))
  
# add the fourth convolutional layer with batch norm, leakyRelu, and a dropout
   discriminator.add(Conv2D(256, kernel_size=3, strides=1, padding="same"))
   discriminator.add(BatchNormalization(momentum=0.8))
   discriminator.add(LeakyReLU(alpha=0.2))
   discriminator.add(Dropout(0.25))
  
# flatten the network and add the output Dense layer with sigmoid activation function
   discriminator.add(Flatten())
   discriminator.add(Dense(1, activation='sigmoid'))
  
# print the model summary
   discriminator.summary()
  
# set the input image shape
   img = Input(shape=img_shape)
  
# run the discriminator model to get the output probability
   probability = discriminator(img)

# return a Model that takes the image as an input and produces the probability output
return Model(img, probability)
```

### Original block 2

```text
def generator_model():

# instantiate a sequential model and name it generator
 generator = Sequential()
# add the Dense layer that has a number of neurons = 128x7x7
 generator.add(Dense(128 * 7 * 7, activation="relu", input_dim=100))
# reshape the image dimensions to 7 x 7 x 128
 generator.add(Reshape((7, 7, 128)))
# upsampling layer to double the size of the image dimensions to 14 x 14
 generator.add(UpSampling2D(size=(2,2)))

# add a convolutional layer to run the convolutional process + batchNorm
 generator.add(Conv2D(128, kernel_size=3, padding="same", activation="relu"))
 generator.add(BatchNormalization(momentum=0.8))

# upsample the image dimensions to 28 x 28
 generator.add(UpSampling2D(size=(2,2)))

# conv + batchNorm layers
# notice that we didn’t add upsampling here because we have the image size of 28 x 28 which
# is equal to the image size in the MNIST dataset. You can adjust this for your own
# problem.
 generator.add(Conv2D(64, kernel_size=3, padding="same", activation="relu"))
 generator.add(BatchNormalization(momentum=0.8))

# convolutional layer with filters = 1
 generator.add(Conv2D(1, kernel_size=3, padding="same", activation="relu"))

# print the model summary
 generator.summary()

# generate the input noise vector of length = 100
# we chose 100 here to create a simple network
 noise = Input(shape=(100,))

# run the generator model to create the fake image
 fake_image = generator(noise)

# return a model that takes the noise vector as an input and outputs the fake image
return Model(noise, fake_image)
```